In [1]:
list.of.packages <- c("haven","labelled","tidyverse","data.table","stargazer")
new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, repos = "http://cran.us.r-project.org")

invisible(lapply(list.of.packages, library, character.only = TRUE))

options(repr.matrix.max.rows=500, repr.matrix.max.cols=1200, message=FALSE, warning=FALSE)  

── Attaching core tidyverse packages ────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attachement du package : ‘data.table’


Les objets suivants sont masqués depuis ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


Les objets suivants sont masqués depuis ‘package:dplyr’:

    between, first, last


L'objet suivant est masqué depuis ‘package:purrr’:

    transp

In [2]:
path <- "./input/IA_2015-16_DHS/IAKR74DT_Children_Stata/IAKR74FL.DTA"

In [3]:
df_birth <- haven::read_dta(path,
                            col_select=c(# Identifiers
                                         "caseid", "bidx","v001","v002","v003","v006","v007","v008",
                                         # Child characteristics
                                         "bord","b0","b1","b2","b3","b4","b5",
                                         # Measurement date (CDC)
                                         "hw17","hw18","hw19",
                                         # Hemoglobin level
                                         "hw56",
                                         # Other health outcomes
                                         "h11","h22","h31","hw72",
                                         # Food
                                         "v409","v410","v411","v411a",
                                         "v412a","v412c","v413",
                                         "v414a","v414e","v414f","v414g","v414i","v414j","v414k","v414l","v414m","v414n","v414o","v414p","v414s","v414t","v414v",
                                         "m4","h42",
                                         # Mother
                                         "v438","v106","s116","v130",
                                         # Wealth
                                         "v190",
                                         # Residence
                                         "v025","v135"
                                   )
                            )
sprintf("%i x %i dataframe", nrow(df_birth), ncol(df_birth))
head(df_birth,2)

[1] "259627 x 54 dataframe"

caseid,v001,v002,v003,v006,v007,v008,v025,v106,v130,v135,v190,v409,v410,v411,v411a,v412a,v412c,v413,v414a,v414e,v414f,v414g,v414i,v414j,v414k,v414l,v414m,v414n,v414o,v414p,v414s,v414t,v414v,v438,bidx,bord,b0,b1,b2,b3,b4,b5,m4,h11,h22,h31,h42,hw17,hw18,hw19,hw56,hw72,s116
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>
01000110 02,10001,10,2,7,2015,1387,1,2,1,1,4,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,1599,1,1,0,6,2011,1338,2,1,24,0,0,2,1,20,7,2015,112,16,4
01000123 02,10001,23,2,7,2015,1387,1,3,1,1,4,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,1537,1,1,0,2,2013,1358,2,1,12,0,0,0,0,20,7,2015,118,-110,NA


# Construct variables

In [4]:
df_birth <- data.frame(df_birth)

In [5]:
df_birth <- df_birth %>%
                    #same variable as in the 2019-2021 survey
                    mutate(b19 = v008 - b3# age in months
                          )

In [6]:
df_birth_var <- df_birth %>%
                    mutate(DHS_round="2015",
                           # Child characteristics
                           Child_female = ifelse(b4==2,1,0),
                           Child_birth_order = bord,
                           Child_alive_age_month = ifelse(b5==1,b19,NA),
                           # Hemoglobin level
                           Child_hemo_level_alti = ifelse(hw56==997,NA,hw56),
                           # Other health outcomes
                           Child_weight_for_height_zscore = ifelse(!is.na(hw72) & hw72!=9996 & hw72!=9997 & hw72!=9998, hw72/100,NA),
                           Child_diarrhea = ifelse(h11==1|h11==2,1,
                                                  ifelse(h11==0,0,NA)),
                           Child_fever = ifelse(h22==1,1,
                                                ifelse(h11==0,0,NA)),
                           Child_cough = ifelse(h31==1|h31==2,1,
                                                  ifelse(h11==0,0,NA)),
                           # Nutrition
                           Child_still_breastfeeding = ifelse(m4==95,1,0),
                           Child_given_plain_water = ifelse(v409==8,NA,v409),
                           Child_given_juice = ifelse(v410==8,NA,v410),
                           Child_given_milk = ifelse(v411==8,NA,v411),
                           Child_given_baby_formula = ifelse(v411a==8,NA,v411a),
                           Child_given_fortified_baby_food = ifelse(v412a==8,NA,v412a),
                           Child_given_soup = ifelse(v412c==8,NA,v412c), 
                           Child_given_other_liquid = ifelse(v413==8,NA,v413), 
                           Child_given_chicken_duck_birds = ifelse(v414a==8,NA,v414a), 
                           Child_given_bread_noodles_grains = ifelse(v414e==8,NA,v414e),
                           Child_given_potatoes_cassava_tubers = ifelse(v414f==8,NA,v414f),
                           Child_given_eggs = ifelse(v414g==8,NA,v414g),
                           Child_given_pumkins_carrots_squash = ifelse(v414i==8,NA,v414i),
                           Child_given_green_vegetables = ifelse(v414j==8,NA,v414j),
                           Child_given_mangoes_papaya_vitaminAfruits = ifelse(v414k==8,NA,v414k),
                           Child_given_other_fruits = ifelse(v414l==8,NA,v414l),
                           Child_given_liver_heart_organs = ifelse(v414m==8,NA,v414m),
                           Child_given_fish_sellfish = ifelse(v414n==8,NA,v414n),
                           Child_given_beans_peas_lentils_nuts = ifelse(v414o==8,NA,v414o),
                           Child_given_cheese_yogurt_milk_products = ifelse(v414p==8,NA,v414p),
                           Child_given_other_solid_semisolid_food = ifelse(v414s==8,NA,v414s),
                           Child_given_other_meat = ifelse(v414t==8,NA,v414t),
                           Child_given_yogurt = ifelse(v414v==8,NA,v414v),
                           Child_iron_supplem_7d = ifelse(h42==1,1,
                                                        ifelse(h42==8|is.na(h42),NA,0)),
                           # Mother
                           Mother_height = ifelse(v438==9994|v438==9995|v438==9996,NA,v438),
                           Mother_no_educ = ifelse(v106==0,1,0),
                           Mother_prim_educ = ifelse(v106==1,1,0),
                           Mother_second_educ = ifelse(v106==2,1,0),
                           Mother_higher_educ = ifelse(v106==3,1,0),                           
                           Mother_ethni_SC =ifelse(s116==1,1,
                                                  ifelse(s116==8|is.na(s116),NA,0)),
                           Mother_ethni_ST =ifelse(s116==2,1,
                                                        ifelse(s116==8|is.na(s116),NA,0)),
                           Mother_ethni_OBC =ifelse(s116==3,1,
                                                        ifelse(s116==8|is.na(s116),NA,0)),
                           Mother_ethni_other =ifelse(s116==4,1,
                                                        ifelse(s116==8|is.na(s116),NA,0)),
                           Mother_hindu = ifelse(v130==1,1,0), 
                           Mother_muslim = ifelse(v130==2,1,0), 
                           Mother_not_hindu_nor_muslim = ifelse(v130!=1 & v130!=2,1,0),
                           # Wealth
                           Wealth_lowest = ifelse(v190==1,1,0),
                           Wealth_second = ifelse(v190==2,1,0),
                           Wealth_middle = ifelse(v190==3,1,0),
                           Wealth_fourth = ifelse(v190==4,1,0),
                           Wealth_highest = ifelse(v190==5,1,0),
                           # Residence
                           Urban = ifelse(v025==1,1,0),
                           Usual_resident = ifelse(v135==2,0,v135),
                           )

## Select variables of interest

In [7]:
df_birth_select <- df_birth_var %>%
                        select(# IDs
                               DHS_round,DHSCLUST=v001,HH_ID=v002,Mother_id=caseid,Child_id=bidx,Resp_nb=v003,
                               Interview_month=v006,Interview_year=v007,
                               Measured_day=hw17,Measured_month=hw18,Measured_year=hw19,
                               Birth_month=b1,Birth_year=b2,
                               # Child characteristics
                               Child_female,Child_birth_order,Child_alive_age_month,
                               # Hemoglobin level
                               Child_hemo_level_alti,
                               # Other health outcomes
                               Child_weight_for_height_zscore,
                               Child_diarrhea,Child_fever,Child_cough, 
                               # Nutrition
                               Child_still_breastfeeding,
                               Child_given_plain_water,Child_given_juice, Child_given_milk, Child_given_baby_formula,
                               Child_given_fortified_baby_food, Child_given_soup, 
                               Child_given_other_liquid, Child_given_chicken_duck_birds, 
                               Child_given_bread_noodles_grains,Child_given_potatoes_cassava_tubers,
                               Child_given_eggs, Child_given_pumkins_carrots_squash, Child_given_green_vegetables,
                               Child_given_mangoes_papaya_vitaminAfruits,Child_given_other_fruits,
                               Child_given_liver_heart_organs,Child_given_fish_sellfish,
                               Child_given_beans_peas_lentils_nuts,Child_given_cheese_yogurt_milk_products,
                               Child_given_other_solid_semisolid_food,Child_given_other_meat,Child_given_yogurt,
                               Child_iron_supplem_7d,
                               # Mother
                               Mother_no_educ,Mother_prim_educ,Mother_second_educ,Mother_higher_educ,
                               Mother_hindu,Mother_muslim,Mother_not_hindu_nor_muslim,
                               Mother_ethni_SC,Mother_ethni_ST,Mother_ethni_OBC,Mother_ethni_other,                               Child_birth_order,
                               Mother_height,
                               # Wealth
                               Wealth_lowest,Wealth_second,Wealth_middle,Wealth_fourth,Wealth_highest,
                               #Residence
                               Urban,Usual_resident
                                )
sprintf("%i x %i dataframe", nrow(df_birth_select), ncol(df_birth_select))
head(df_birth_select, 2)

[1] "259627 x 64 dataframe"

,DHS_round,DHSCLUST,HH_ID,Mother_id,Child_id,Resp_nb,Interview_month,Interview_year,Measured_day,Measured_month,Measured_year,Birth_month,Birth_year,Child_female,Child_birth_order,Child_alive_age_month,Child_hemo_level_alti,Child_weight_for_height_zscore,Child_diarrhea,Child_fever,Child_cough,Child_still_breastfeeding,Child_given_plain_water,Child_given_juice,Child_given_milk,Child_given_baby_formula,Child_given_fortified_baby_food,Child_given_soup,Child_given_other_liquid,Child_given_chicken_duck_birds,Child_given_bread_noodles_grains,Child_given_potatoes_cassava_tubers,Child_given_eggs,Child_given_pumkins_carrots_squash,Child_given_green_vegetables,Child_given_mangoes_papaya_vitaminAfruits,Child_given_other_fruits,Child_given_liver_heart_organs,Child_given_fish_sellfish,Child_given_beans_peas_lentils_nuts,Child_given_cheese_yogurt_milk_products,Child_given_other_solid_semisolid_food,Child_given_other_meat,Child_given_yogurt,Child_iron_supplem_7d,Mother_no_educ,Mother_prim_educ,Mother_second_educ,Mother_higher_educ,Mother_hindu,Mother_muslim,Mother_not_hindu_nor_muslim,Mother_ethni_SC,Mother_ethni_ST,Mother_ethni_OBC,Mother_ethni_other,Mother_height,Wealth_lowest,Wealth_second,Wealth_middle,Wealth_fourth,Wealth_highest,Urban,Usual_resident
,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2015,10001,10,01000110 02,1,2,7,2015,20,7,2015,6,2011,1,1,49,112,0.16,0,0,1,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,1,0,0,1,0,1,0,0,0,0,0,1,1599,0,0,0,1,0,1,1
2,2015,10001,23,01000123 02,1,2,7,2015,20,7,2015,2,2013,1,1,29,118,-1.10,0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,0,0,0,0,1,1,0,0,NA,NA,NA,NA,1537,0,0,0,1,0,1,1


## Add bednet & water access in HH members file

In [8]:
path_HH <- "../IA_2015-16_DHS/IAPR74DT_HHmembers_Stata/IAPR74FL.DTA"
df_HH <- haven::read_dta(path_HH,
                         col_select=c("hvidx","hc0","hv001","hv002","hv003",
                                      # Bednet
                                      "hml12",
                                      # Water
                                      "hv204",
                                      # Air conditioner/cooler
                                      "sh37q",
                                      # WASH
                                      "hv237","hv205","hv201"
                                     )
                        )
sprintf("%i x %i dataframe", nrow(df_HH), ncol(df_HH))
head(df_HH, 2)

[1] "2869043 x 11 dataframe"

hvidx,hv001,hv002,hv003,hv201,hv204,hv205,hv237,sh37q,hc0,hml12
<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl>,<dbl+lbl>
1,10001,1,3,11,996,12,0,0,NA,0
2,10001,1,3,11,996,12,0,0,NA,0


In [9]:
df_watbed <- df_HH %>%
                mutate(Bednet_slept = ifelse(hml12==0,0,1),
                       HH_time_water_source = ifelse(hv204==996,0,
                                                    ifelse(hv204==998,NA,hv204)),
                       HH_air_conditioner = sh37q,
                       HH_treat_water = ifelse(hv237==8,NA,hv237),
                       HH_no_toilet = ifelse(hv205==30|hv205==31,1,0),
                       HH_well_water = ifelse(hv201==21|hv201==31|hv201==32,1,ifelse(!is.na(hv201),0,NA))
                       )%>%
                select(hvidx,hv001,hv002,
                       Bednet_slept,
                       HH_time_water_source,
                       HH_air_conditioner,
                       HH_treat_water,HH_no_toilet,HH_well_water)
sprintf("%i x %i dataframe", nrow(df_watbed), ncol(df_watbed))
head(df_watbed, 2)

[1] "2869043 x 9 dataframe"

hvidx,hv001,hv002,Bednet_slept,HH_time_water_source,HH_air_conditioner,HH_treat_water,HH_no_toilet,HH_well_water
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl>
1,10001,1,0,0,0,0,0,0
2,10001,1,0,0,0,0,0,0


In [10]:
df_birth_select_watbed <- df_birth_select %>%
                            left_join(df_watbed,by=c("DHSCLUST"="hv001","HH_ID"="hv002","Resp_nb"="hvidx"))
sprintf("%i x %i dataframe", nrow(df_birth_select_watbed), ncol(df_birth_select_watbed))
head(df_birth_select_watbed, 2)

[1] "259627 x 70 dataframe"

,DHS_round,DHSCLUST,HH_ID,Mother_id,Child_id,Resp_nb,Interview_month,Interview_year,Measured_day,Measured_month,Measured_year,Birth_month,Birth_year,Child_female,Child_birth_order,Child_alive_age_month,Child_hemo_level_alti,Child_weight_for_height_zscore,Child_diarrhea,Child_fever,Child_cough,Child_still_breastfeeding,Child_given_plain_water,Child_given_juice,Child_given_milk,Child_given_baby_formula,Child_given_fortified_baby_food,Child_given_soup,Child_given_other_liquid,Child_given_chicken_duck_birds,Child_given_bread_noodles_grains,Child_given_potatoes_cassava_tubers,Child_given_eggs,Child_given_pumkins_carrots_squash,Child_given_green_vegetables,Child_given_mangoes_papaya_vitaminAfruits,Child_given_other_fruits,Child_given_liver_heart_organs,Child_given_fish_sellfish,Child_given_beans_peas_lentils_nuts,Child_given_cheese_yogurt_milk_products,Child_given_other_solid_semisolid_food,Child_given_other_meat,Child_given_yogurt,Child_iron_supplem_7d,Mother_no_educ,Mother_prim_educ,Mother_second_educ,Mother_higher_educ,Mother_hindu,Mother_muslim,Mother_not_hindu_nor_muslim,Mother_ethni_SC,Mother_ethni_ST,Mother_ethni_OBC,Mother_ethni_other,Mother_height,Wealth_lowest,Wealth_second,Wealth_middle,Wealth_fourth,Wealth_highest,Urban,Usual_resident,Bednet_slept,HH_time_water_source,HH_air_conditioner,HH_treat_water,HH_no_toilet,HH_well_water
,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl>
1,2015,10001,10,01000110 02,1,2,7,2015,20,7,2015,6,2011,1,1,49,112,0.16,0,0,1,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,1,0,0,1,0,1,0,0,0,0,0,1,1599,0,0,0,1,0,1,1,0,0,0,1,0,0
2,2015,10001,23,01000123 02,1,2,7,2015,20,7,2015,2,2013,1,1,29,118,-1.10,0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,0,0,0,0,1,1,0,0,NA,NA,NA,NA,1537,0,0,0,1,0,1,1,0,0,0,1,0,0


## Add state and district according to 2011 census

In [11]:
df_2011_census <- read_csv("./interm/Matching_DHSclusters_2015_census_2011.csv",show_col_types = FALSE)
sprintf("%i x %i dataframe", nrow(df_2011_census), ncol(df_2011_census))
head(df_2011_census, 2)

[1] "28526 x 3 dataframe"

DHSCLUST,State_2011,District_2011
<dbl>,<chr>,<chr>
310502,Tamil Nadu,Thiruvallur
310190,Tamil Nadu,Thiruvallur


In [12]:
df_birth_census <- df_birth_select_watbed %>%
                                left_join(df_2011_census,by="DHSCLUST")%>%
                                relocate(c(State_2011,District_2011),.after=DHSCLUST)
sprintf("%i x %i dataframe", nrow(df_birth_census), ncol(df_birth_census))
head(df_birth_census, 2)

[1] "259627 x 72 dataframe"

,DHS_round,DHSCLUST,State_2011,District_2011,HH_ID,Mother_id,Child_id,Resp_nb,Interview_month,Interview_year,Measured_day,Measured_month,Measured_year,Birth_month,Birth_year,Child_female,Child_birth_order,Child_alive_age_month,Child_hemo_level_alti,Child_weight_for_height_zscore,Child_diarrhea,Child_fever,Child_cough,Child_still_breastfeeding,Child_given_plain_water,Child_given_juice,Child_given_milk,Child_given_baby_formula,Child_given_fortified_baby_food,Child_given_soup,Child_given_other_liquid,Child_given_chicken_duck_birds,Child_given_bread_noodles_grains,Child_given_potatoes_cassava_tubers,Child_given_eggs,Child_given_pumkins_carrots_squash,Child_given_green_vegetables,Child_given_mangoes_papaya_vitaminAfruits,Child_given_other_fruits,Child_given_liver_heart_organs,Child_given_fish_sellfish,Child_given_beans_peas_lentils_nuts,Child_given_cheese_yogurt_milk_products,Child_given_other_solid_semisolid_food,Child_given_other_meat,Child_given_yogurt,Child_iron_supplem_7d,Mother_no_educ,Mother_prim_educ,Mother_second_educ,Mother_higher_educ,Mother_hindu,Mother_muslim,Mother_not_hindu_nor_muslim,Mother_ethni_SC,Mother_ethni_ST,Mother_ethni_OBC,Mother_ethni_other,Mother_height,Wealth_lowest,Wealth_second,Wealth_middle,Wealth_fourth,Wealth_highest,Urban,Usual_resident,Bednet_slept,HH_time_water_source,HH_air_conditioner,HH_treat_water,HH_no_toilet,HH_well_water
,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl>
1,2015,10001,Andaman & Nicobar Island,South Andaman,10,01000110 02,1,2,7,2015,20,7,2015,6,2011,1,1,49,112,0.16,0,0,1,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,1,0,0,1,0,1,0,0,0,0,0,1,1599,0,0,0,1,0,1,1,0,0,0,1,0,0
2,2015,10001,Andaman & Nicobar Island,South Andaman,23,01000123 02,1,2,7,2015,20,7,2015,2,2013,1,1,29,118,-1.10,0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,0,0,0,0,1,1,0,0,NA,NA,NA,NA,1537,0,0,0,1,0,1,1,0,0,0,1,0,0


## Add cells ID

In [13]:
df_cells_coord_01 <- read_csv("../matching_grid_cells_DHS_clusters/output/DHS_15_matching_clust_cells_0.1.csv",show_col_types = FALSE)
sprintf("%i x %i dataframe", nrow(df_cells_coord_01), ncol(df_cells_coord_01))
head(df_cells_coord_01, 2)

[1] "28395 x 3 dataframe"

DHSCLUST,cell_x,cell_y
<dbl>,<dbl>,<dbl>
310502,80.0,13.3
310190,79.8,13.1


In [14]:
nrow(df_cells_coord_01%>%distinct(cell_x,cell_y))

[1] 12784

In [15]:
df_cells_coord_025 <- read_csv("../matching_grid_cells_DHS_clusters/output/DHS_15_matching_clust_cells_0.25.csv",show_col_types = FALSE)
sprintf("%i x %i dataframe", nrow(df_cells_coord_025), ncol(df_cells_coord_025))
head(df_cells_coord_025, 2)

[1] "28395 x 3 dataframe"

DHSCLUST,cell_x,cell_y
<dbl>,<dbl>,<dbl>
310502,80.00,13.25
310190,79.75,13.00


In [16]:
df_birth_cells <- df_birth_census %>%
                                left_join(df_cells_coord_01%>%rename(cell_x_01=cell_x,cell_y_01=cell_y),
                                          by="DHSCLUST")%>%
                                left_join(df_cells_coord_025%>%rename(cell_x_025=cell_x,cell_y_025=cell_y),
                                          by="DHSCLUST")%>%
                                relocate(c(cell_x_01,cell_y_01,cell_x_025,cell_y_025),.after=District_2011)
sprintf("%i x %i dataframe", nrow(df_birth_cells), ncol(df_birth_cells))
head(df_birth_cells, 2)

[1] "259627 x 76 dataframe"

,DHS_round,DHSCLUST,State_2011,District_2011,cell_x_01,cell_y_01,cell_x_025,cell_y_025,HH_ID,Mother_id,Child_id,Resp_nb,Interview_month,Interview_year,Measured_day,Measured_month,Measured_year,Birth_month,Birth_year,Child_female,Child_birth_order,Child_alive_age_month,Child_hemo_level_alti,Child_weight_for_height_zscore,Child_diarrhea,Child_fever,Child_cough,Child_still_breastfeeding,Child_given_plain_water,Child_given_juice,Child_given_milk,Child_given_baby_formula,Child_given_fortified_baby_food,Child_given_soup,Child_given_other_liquid,Child_given_chicken_duck_birds,Child_given_bread_noodles_grains,Child_given_potatoes_cassava_tubers,Child_given_eggs,Child_given_pumkins_carrots_squash,Child_given_green_vegetables,Child_given_mangoes_papaya_vitaminAfruits,Child_given_other_fruits,Child_given_liver_heart_organs,Child_given_fish_sellfish,Child_given_beans_peas_lentils_nuts,Child_given_cheese_yogurt_milk_products,Child_given_other_solid_semisolid_food,Child_given_other_meat,Child_given_yogurt,Child_iron_supplem_7d,Mother_no_educ,Mother_prim_educ,Mother_second_educ,Mother_higher_educ,Mother_hindu,Mother_muslim,Mother_not_hindu_nor_muslim,Mother_ethni_SC,Mother_ethni_ST,Mother_ethni_OBC,Mother_ethni_other,Mother_height,Wealth_lowest,Wealth_second,Wealth_middle,Wealth_fourth,Wealth_highest,Urban,Usual_resident,Bednet_slept,HH_time_water_source,HH_air_conditioner,HH_treat_water,HH_no_toilet,HH_well_water
,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl>
1,2015,10001,Andaman & Nicobar Island,South Andaman,92.7,11.7,92.75,11.75,10,01000110 02,1,2,7,2015,20,7,2015,6,2011,1,1,49,112,0.16,0,0,1,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,1,0,0,1,0,1,0,0,0,0,0,1,1599,0,0,0,1,0,1,1,0,0,0,1,0,0
2,2015,10001,Andaman & Nicobar Island,South Andaman,92.7,11.7,92.75,11.75,23,01000123 02,1,2,7,2015,20,7,2015,2,2013,1,1,29,118,-1.10,0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,0,0,0,0,1,1,0,0,NA,NA,NA,NA,1537,0,0,0,1,0,1,1,0,0,0,1,0,0


In [17]:
nrow(df_birth_cells%>%distinct(cell_x_01,cell_y_01))

[1] 12747

## Convert measured date in CDC

In [18]:
df_birth_CDC <- df_birth_cells %>%
                        mutate(Measured_date = as.Date(paste(Measured_year, Measured_month, Measured_day, sep = "-")),
                               Measured_date_CDC = as.numeric(Measured_date - as.Date("1900-01-01")))%>%
                        relocate(c(Measured_date_CDC,Measured_date),.before=Measured_day)
sprintf("%i x %i dataframe", nrow(df_birth_CDC), ncol(df_birth_CDC))
head(df_birth_CDC, 2)

[1] "259627 x 78 dataframe"

,DHS_round,DHSCLUST,State_2011,District_2011,cell_x_01,cell_y_01,cell_x_025,cell_y_025,HH_ID,Mother_id,Child_id,Resp_nb,Interview_month,Interview_year,Measured_date_CDC,Measured_date,Measured_day,Measured_month,Measured_year,Birth_month,Birth_year,Child_female,Child_birth_order,Child_alive_age_month,Child_hemo_level_alti,Child_weight_for_height_zscore,Child_diarrhea,Child_fever,Child_cough,Child_still_breastfeeding,Child_given_plain_water,Child_given_juice,Child_given_milk,Child_given_baby_formula,Child_given_fortified_baby_food,Child_given_soup,Child_given_other_liquid,Child_given_chicken_duck_birds,Child_given_bread_noodles_grains,Child_given_potatoes_cassava_tubers,Child_given_eggs,Child_given_pumkins_carrots_squash,Child_given_green_vegetables,Child_given_mangoes_papaya_vitaminAfruits,Child_given_other_fruits,Child_given_liver_heart_organs,Child_given_fish_sellfish,Child_given_beans_peas_lentils_nuts,Child_given_cheese_yogurt_milk_products,Child_given_other_solid_semisolid_food,Child_given_other_meat,Child_given_yogurt,Child_iron_supplem_7d,Mother_no_educ,Mother_prim_educ,Mother_second_educ,Mother_higher_educ,Mother_hindu,Mother_muslim,Mother_not_hindu_nor_muslim,Mother_ethni_SC,Mother_ethni_ST,Mother_ethni_OBC,Mother_ethni_other,Mother_height,Wealth_lowest,Wealth_second,Wealth_middle,Wealth_fourth,Wealth_highest,Urban,Usual_resident,Bednet_slept,HH_time_water_source,HH_air_conditioner,HH_treat_water,HH_no_toilet,HH_well_water
,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl>
1,2015,10001,Andaman & Nicobar Island,South Andaman,92.7,11.7,92.75,11.75,10,01000110 02,1,2,7,2015,42203,2015-07-20,20,7,2015,6,2011,1,1,49,112,0.16,0,0,1,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,1,0,0,1,0,1,0,0,0,0,0,1,1599,0,0,0,1,0,1,1,0,0,0,1,0,0
2,2015,10001,Andaman & Nicobar Island,South Andaman,92.7,11.7,92.75,11.75,23,01000123 02,1,2,7,2015,42203,2015-07-20,20,7,2015,2,2013,1,1,29,118,-1.10,0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,0,0,0,0,1,1,0,0,NA,NA,NA,NA,1537,0,0,0,1,0,1,1,0,0,0,1,0,0


## Final selection

In [19]:
df_birth_final <- df_birth_CDC %>%
                            filter(!is.na(Measured_date) & !is.na(cell_x_01))
sprintf("%i x %i dataframe", nrow(df_birth_final), ncol(df_birth_final))
head(df_birth_final, 2)

[1] "243371 x 78 dataframe"

,DHS_round,DHSCLUST,State_2011,District_2011,cell_x_01,cell_y_01,cell_x_025,cell_y_025,HH_ID,Mother_id,Child_id,Resp_nb,Interview_month,Interview_year,Measured_date_CDC,Measured_date,Measured_day,Measured_month,Measured_year,Birth_month,Birth_year,Child_female,Child_birth_order,Child_alive_age_month,Child_hemo_level_alti,Child_weight_for_height_zscore,Child_diarrhea,Child_fever,Child_cough,Child_still_breastfeeding,Child_given_plain_water,Child_given_juice,Child_given_milk,Child_given_baby_formula,Child_given_fortified_baby_food,Child_given_soup,Child_given_other_liquid,Child_given_chicken_duck_birds,Child_given_bread_noodles_grains,Child_given_potatoes_cassava_tubers,Child_given_eggs,Child_given_pumkins_carrots_squash,Child_given_green_vegetables,Child_given_mangoes_papaya_vitaminAfruits,Child_given_other_fruits,Child_given_liver_heart_organs,Child_given_fish_sellfish,Child_given_beans_peas_lentils_nuts,Child_given_cheese_yogurt_milk_products,Child_given_other_solid_semisolid_food,Child_given_other_meat,Child_given_yogurt,Child_iron_supplem_7d,Mother_no_educ,Mother_prim_educ,Mother_second_educ,Mother_higher_educ,Mother_hindu,Mother_muslim,Mother_not_hindu_nor_muslim,Mother_ethni_SC,Mother_ethni_ST,Mother_ethni_OBC,Mother_ethni_other,Mother_height,Wealth_lowest,Wealth_second,Wealth_middle,Wealth_fourth,Wealth_highest,Urban,Usual_resident,Bednet_slept,HH_time_water_source,HH_air_conditioner,HH_treat_water,HH_no_toilet,HH_well_water
,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl>
1,2015,10001,Andaman & Nicobar Island,South Andaman,92.7,11.7,92.75,11.75,10,01000110 02,1,2,7,2015,42203,2015-07-20,20,7,2015,6,2011,1,1,49,112,0.16,0,0,1,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,1,0,0,1,0,1,0,0,0,0,0,1,1599,0,0,0,1,0,1,1,0,0,0,1,0,0
2,2015,10001,Andaman & Nicobar Island,South Andaman,92.7,11.7,92.75,11.75,23,01000123 02,1,2,7,2015,42203,2015-07-20,20,7,2015,2,2013,1,1,29,118,-1.10,0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,0,0,0,0,1,1,0,0,NA,NA,NA,NA,1537,0,0,0,1,0,1,1,0,0,0,1,0,0


## Save

In [20]:
saveRDS(df_birth_final,"./output/df_children_2015_16.rds")